### CrisisLens AI: Real-Time Disaster Intelligence for Southeast Asia

#### Multi-source verified crisis data + role-based Gemma AI, live on web dashboard and Telegram


CrisisLens AI turns scattered, unverified disaster reports into role-specific, actionable guidance for citizens, government officials, emergency responders, and tourists across Indonesia and Southeast Asia.

A single Kaggle Notebook pipeline ingests live reports from RSS/JSON feeds, NewsAPI, and official sources (USGS, BMKG, WHO, ReliefWeb), filters them for relevance, and uses Gemma (gemma-4-31b-it) to extract structured crisis data — type, severity, location, magnitude, casualties. Reports of the same event from different sources are semantically deduplicated with multilingual SBERT and fused into a single verified record, scored by a 5-factor confidence engine that rewards official-source corroboration and geocoding success.

That verified event data powers three live, working interfaces built from the same pipeline: an inline Gemma AI assistant and real-time crisis map directly in the notebook, a public Streamlit web dashboard (bilingual English/Bahasa Indonesia, with analytics and a searchable incident table), and a Telegram bot offering role selection, live status, and free-text Gemma Q&A — reaching users with no app install required.

Every assistant across all three interfaces shares the same role-grounded prompting, so a government official gets the same analytical guidance whether they're on the dashboard or messaging the bot from their phone.

In [1]:
# ======================================================================
# CELL 1: Setup + API Keys + Model Discovery
# ======================================================================
import subprocess, sys, os, json, pickle, time, asyncio
from datetime import datetime, timezone, timedelta
from collections import Counter
from kaggle_secrets import UserSecretsClient

# Install packages
PACKAGES = [
    "google-genai", "feedparser", "sentence-transformers", "transformers", "torch",
    "geopy", "python-telegram-bot", "langdetect", "spacy", "requests",
    "streamlit", "streamlit-folium", "folium", "plotly", "tqdm", "pyngrok", "nest-asyncio"
]
print("Installing packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES, check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "xx_ent_wiki_sm", "-q"], check=True)

# Load secrets
from google import genai
from google.genai import types
secrets = UserSecretsClient()
def load_secret(name, required=True):
    try:
        return secrets.get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"Required secret '{name}' not found")
        return None

os.environ["GOOGLE_API_KEY"]      = load_secret("GOOGLE_API_KEY", required=True)
os.environ["NEWSAPI_KEY"]         = load_secret("NEWSAPI_KEY", required=False) or ""
os.environ["TELEGRAM_BOT_TOKEN"]  = load_secret("TELEGRAM_BOT_TOKEN", required=False) or ""
os.environ["TELEGRAM_CHAT_ID"]    = load_secret("TELEGRAM_CHAT_ID", required=False) or ""
os.environ["NGROK_AUTHTOKEN"]     = load_secret("NGROK_AUTHTOKEN", required=False) or ""

client = genai.Client()

# Model discovery
def find_working_model(candidates, max_tokens=300):
    for name in candidates:
        try:
            response = client.models.generate_content(
                model=name, contents="Reply with only the word: OK",
                config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=max_tokens)
            )
            if response.text and response.text.strip():
                print(f"   {name}: {response.text.strip()}")
                return name
        except Exception as e:
            print(f"   {name}: {e}")
    return None

print("Finding working models...")
FAST_MODEL = find_working_model(["gemma-4-31b-it", "gemma-4-26b-a4b-it"])
DEEP_MODEL = find_working_model(["gemma-4-31b-it"])

if not FAST_MODEL or not DEEP_MODEL:
    raise RuntimeError("No working models found")

print(f"\n FAST_MODEL = '{FAST_MODEL}'")
print(f" DEEP_MODEL = '{DEEP_MODEL}'")
print(f" Setup complete - {len(PACKAGES)} packages installed")

Installing packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.5/530.5 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 76.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Finding working models...
   gemma-4-31b-it: OK
   gemma-4-31b-it: OK

 FAST_MODE

In [3]:
# ======================================================================
# CELL 2: Data Sources + Collection Functions
# ======================================================================
import feedparser, requests, html, re
from tqdm import tqdm

# 9 Data Sources
SOURCES = [
    {
        "name": "USGS",
        "url": "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_day.atom?minlatitude=-11&maxlatitude=38&minlongitude=60&maxlongitude=141",
        "format": "rss", "type": "official", "trust": 1.00, "hint": "earthquake", "lang": "en"
    },
    {
        "name": "BMKG", "url": "https://data.bmkg.go.id/DataMKG/TEWS/autogempa.json",
        "format": "json", "type": "official", "trust": 1.00, "hint": "earthquake", "lang": "id"
    },
    {
        "name": "WHO", "url": "https://www.who.int/rss-feeds/news-english.xml",
        "format": "rss", "type": "official", "trust": 0.95, "hint": "disease", "lang": "en"
    },
    {
        "name": "ReliefWeb", "url": "https://reliefweb.int/updates/rss.xml",
        "format": "rss", "type": "official", "trust": 0.95, "hint": "multi", "lang": "en"
    },
    {
        "name": "BBC", "url": "https://feeds.bbci.co.uk/news/world/rss.xml",
        "format": "rss", "type": "news", "trust": 0.90, "hint": "multi", "lang": "en"
    },
    {
        "name": "AlJazeera", "url": "https://www.aljazeera.com/xml/rss/all.xml",
        "format": "rss", "type": "news", "trust": 0.88, "hint": "multi", "lang": "en"
    },
    {
        "name": "Antara", "url": "https://www.antaranews.com/rss/terkini.xml",
        "format": "rss", "type": "news", "trust": 0.85, "hint": "multi", "lang": "id"
    },
    {
        "name": "GoogleNewsID",
        "url": "https://news.google.com/rss/search?q=bencana+Indonesia&hl=id&gl=ID&ceid=ID:id",
        "format": "rss", "type": "news", "trust": 0.82, "hint": "multi", "lang": "id"
    },
    {
        "name": "GoogleNewsAsia",
        "url": "https://news.google.com/rss/search?q=Asia+disaster&hl=en-US&gl=US&ceid=US:en",
        "format": "rss", "type": "news", "trust": 0.82, "hint": "multi", "lang": "en"
    }
]

HEADERS = {"User-Agent": "CrisisLens-AI/2.0 (+Kaggle)"}

def collect_rss(source):
    reports = []
    try:
        feed = feedparser.parse(source["url"], request_headers=HEADERS)
        for entry in feed.entries[:50]:
            title   = html.unescape(entry.get("title", ""))
            summary = html.unescape(re.sub("<.*?>", " ", entry.get("summary", "")))
            text    = re.sub(r"\s+", " ", f"{title}. {summary}".strip())
            if len(text) < 25:
                continue
            reports.append({
                "source_name": source["name"], "source_type": source["type"],
                "trust_score": source["trust"], "raw_text": text,
                "source_url": entry.get("link", ""), "crisis_hint": source["hint"],
                "language": source["lang"],
                "published_at": entry.get("published", datetime.now(timezone.utc).isoformat())
            })
    except Exception as e:
        print(f"  {source['name']} failed: {e}")
    return reports

def collect_json(source):
    reports = []
    try:
        r = requests.get(source["url"], headers=HEADERS, timeout=20)
        r.raise_for_status()
        data = r.json()
        if source["name"] == "BMKG":
            gempa = data.get("Infogempa", {}).get("gempa", {})
            if gempa:
                text = (f"Gempa bumi Magnitude {gempa.get('Magnitude','?')} "
                        f"di {gempa.get('Wilayah','Unknown')}. "
                        f"Kedalaman {gempa.get('Kedalaman','?')}. "
                        f"{gempa.get('Potensi','')}")
                reports.append({
                    "source_name": "BMKG", "source_type": "official", "trust_score": 1.0,
                    "raw_text": text, "source_url": "https://www.bmkg.go.id",
                    "crisis_hint": "earthquake", "language": "id",
                    "published_at": datetime.now(timezone.utc).isoformat()
                })
    except Exception as e:
        print(f"  {source['name']} failed: {e}")
    return reports

def collect_newsapi(hours_back=6):
    reports = []
    api_key = os.environ.get("NEWSAPI_KEY")
    if not api_key:
        print("  NewsAPI key missing - skipping")
        return reports
    try:
        from_time = (datetime.now(timezone.utc) - timedelta(hours=hours_back)).isoformat()
        query = "(earthquake OR flood OR landslide OR tsunami OR volcano OR wildfire OR cyclone OR disaster)"
        r = requests.get("https://newsapi.org/v2/everything", params={
            "q": query, "language": "en", "sortBy": "publishedAt",
            "from": from_time, "pageSize": 100, "apiKey": api_key
        }, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            for article in r.json().get("articles", [])[:50]:
                text = f"{article.get('title', '')}. {article.get('description', '')}".strip()
                if len(text) > 25:
                    reports.append({
                        "source_name": "NewsAPI", "source_type": "news", "trust_score": 0.75,
                        "raw_text": text, "source_url": article.get("url", ""),
                        "crisis_hint": "multi", "language": "en",
                        "published_at": article.get("publishedAt", "")
                    })
        else:
            print(f"  NewsAPI error: {r.status_code} - {r.text[:100]}")
    except Exception as e:
        print(f"  NewsAPI failed: {e}")
    return reports

print(f"  Collection functions ready - {len(SOURCES)} sources configured")

  Collection functions ready - 9 sources configured


In [4]:
# ======================================================================
# CELL 3: Run Collection (all 9 sources)
# ======================================================================
import pandas as pd

print("=" * 70)
print(" CrisisLens AI 2.0 - Data Collection")
print("=" * 70)

raw_reports = []

print(" Collecting from RSS/JSON sources...")
for source in tqdm(SOURCES, desc="Sources"):
    if source["format"] == "rss":
        reports = collect_rss(source)
    elif source["format"] == "json":
        reports = collect_json(source)
    else:
        reports = []

    status = "" if reports else " "
    print(f"  {status} {source['name']:<20} {len(reports)} reports")
    raw_reports.extend(reports)

print("\n  Collecting from NewsAPI...")
newsapi_reports = collect_newsapi(hours_back=6)
status = " " if newsapi_reports else "  "
print(f"  {status} NewsAPI               {len(newsapi_reports)} reports")
raw_reports.extend(newsapi_reports)

# URL-level deduplication
seen, unique = set(), []
for r in raw_reports:
    key = r.get("source_url", "").strip() or r["raw_text"][:80]
    if key not in seen:
        seen.add(key)
        unique.append(r)
raw_reports = unique

print(f"\n  Total unique reports: {len(raw_reports)}")

# Save
os.makedirs("/kaggle/working", exist_ok=True)
pd.DataFrame(raw_reports).to_csv("/kaggle/working/raw_reports.csv", index=False)
with open("/kaggle/working/raw_reports.pkl", "wb") as f:
    pickle.dump(raw_reports, f)

print("\nSample reports:")
for i, r in enumerate(raw_reports[:3], 1):
    print(f"{i}. [{r['source_name']}] {r['raw_text'][:100]}...")

print(f"\n  Raw collection complete - {len(raw_reports)} reports saved")

 CrisisLens AI 2.0 - Data Collection


Sources:  22%|██▏       | 2/9 [00:00<00:00, 17.34it/s]

   USGS                 15 reports
   BMKG                 1 reports
   WHO                  25 reports


Sources:  44%|████▍     | 4/9 [00:00<00:00, 12.86it/s]

    ReliefWeb            0 reports
   BBC                  29 reports
   AlJazeera            25 reports


...

   Antara               50 reports


Sources:  89%|████████▉ | 8/9 [00:01<00:00,  3.75it/s]

   GoogleNewsID         50 reports


Sources: 100%|██████████| 9/9 [00:02<00:00,  4.03it/s]

   GoogleNewsAsia       50 reports

     NewsAPI               0 reports

  Total unique reports: 245

Sample reports:
1. [USGS] M 5.9 - 42 km NNW of Te Anau, New Zealand. PAGER - GREEN ShakeMap - V DYFI? - IV Time 2026-07-16 09:...
2. [USGS] M 5.0 - 141 km ENE of Hicks Bay, New Zealand. Time 2026-07-16 08:57:58 UTC 2026-07-16 08:57:58 UTC a...
3. [USGS] M 5.5 - Volcano Islands, Japan region. PAGER - GREEN ShakeMap - I Time 2026-07-16 07:49:09 UTC 2026-...

  Raw collection complete - 245 reports saved


In [5]:
# ======================================================================
# CELL 4: Geographic + Crisis Filtering
# ======================================================================

TARGET_COUNTRY_KEYWORDS = {
    "indonesia":   ["indonesia","jakarta","bandung","surabaya","bali","java","sumatra",
                    "kalimantan","sulawesi","papua","aceh","yogyakarta","lombok"],
    "india":       ["india","delhi","mumbai","kolkata","chennai","bengaluru","hyderabad"],
    "philippines": ["philippines","manila","cebu","mindanao","luzon","davao"],
    "pakistan":    ["pakistan","karachi","lahore","islamabad","quetta"],
    "bangladesh":  ["bangladesh","dhaka","chittagong","sylhet"],
    "vietnam":     ["vietnam","hanoi","ho chi minh","da nang"],
    "thailand":    ["thailand","bangkok","chiang mai","phuket"],
    "myanmar":     ["myanmar","burma","yangon","naypyidaw"],
    "nepal":       ["nepal","kathmandu","pokhara"],
    "sri lanka":   ["sri lanka","colombo","kandy"]
}

DISASTER_KEYWORDS = {
    "earthquake","quake","aftershock","flood","flash flood","landslide","mudslide",
    "tsunami","cyclone","typhoon","storm","volcano","eruption","wildfire",
    "disaster","emergency","evacuation","outbreak","epidemic","pandemic",
    "gempa","banjir","longsor","erupsi","kebakaran","darurat","evakuasi","bencana"
}

ALL_LOCATION_WORDS  = set(w for vals in TARGET_COUNTRY_KEYWORDS.values() for w in vals)
ALWAYS_PASS_SOURCES = {"BMKG"}

def is_target_region(text, source_hint="multi", source_name=""):
    if source_name in ALWAYS_PASS_SOURCES:
        return True
    text = text.lower()
    if "papua new guinea" in text:
        return False
    has_location = any(w in text for w in ALL_LOCATION_WORDS)
    if source_hint in {"earthquake", "disease"}:
        return has_location
    has_disaster = any(w in text for w in DISASTER_KEYWORDS)
    return has_location and has_disaster

print("  Filtering for target region + crisis relevance...")
crisis_reports = []

for report in raw_reports:
    if is_target_region(report["raw_text"], report["crisis_hint"], report["source_name"]):
        report["cleaned_text"] = re.sub(r'[^\w\s.,;:!?-]', '', report["raw_text"])
        report["entities"]     = {}
        crisis_reports.append(report)

print(f"  Filtering results:")
print(f"  Raw reports     : {len(raw_reports)}")
print(f"  Crisis relevant : {len(crisis_reports)}")
filtered_out = len(raw_reports) - len(crisis_reports)
pct = 100 * filtered_out / len(raw_reports) if raw_reports else 0
print(f"  Filtered out    : {filtered_out} ({pct:.1f}%)")

with open("/kaggle/working/crisis_reports.pkl", "wb") as f:
    pickle.dump(crisis_reports, f)

print(f"\n  Geographic filtering complete - {len(crisis_reports)} crisis-relevant reports")

  Filtering for target region + crisis relevance...
  Filtering results:
  Raw reports     : 245
  Crisis relevant : 35
  Filtered out    : 210 (85.7%)

  Geographic filtering complete - 35 crisis-relevant reports


In [6]:
# ======================================================================
# CELL 5: Binary Filter + Classification Setup
# ======================================================================
from transformers import pipeline as hf_pipeline

# Load multilingual zero-shot crisis classifier
print("  Loading binary filter model...")
try:
    binary_filter = hf_pipeline(
        "zero-shot-classification",
        model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
        device=0 if __import__("torch").cuda.is_available() else -1
    )
    CRISIS_LABELS    = ["crisis or disaster", "not crisis"]
    CRISIS_THRESHOLD = 0.65
    print(f"  Binary filter loaded | threshold={CRISIS_THRESHOLD}")
except Exception as e:
    print(f"  Binary filter failed to load: {e}")
    binary_filter = None

# Rate limiter
class RateLimiter:
    def __init__(self, calls_per_minute=15):
        self.min_interval = 60.0 / calls_per_minute
        self.last_call    = 0

    def wait(self):
        now    = time.time()
        diff   = now - self.last_call
        if diff < self.min_interval:
            sleep_time = self.min_interval - diff
            print(f"    rate limit: sleeping {sleep_time:.1f}s")
            time.sleep(sleep_time)
        self.last_call = time.time()

limiter = RateLimiter(calls_per_minute=15)

# Classification function
def classify_and_extract(report, retries=2):
    prompt = f"""You are a crisis intelligence classifier and data extractor.
The text may be in any language — read it directly, do not translate first.

Text: {report['cleaned_text'][:1200]}
Source: {report['source_name']} ({report['source_type']}, hint: {report.get('crisis_hint','unknown')})

Return ONLY valid JSON, no markdown fences, no explanation:
{{
  "crisis_type": "earthquake|flood|wildfire|storm|volcano|disease|conflict|industrial|other",
  "sub_type": "more specific type or null",
  "confidence": 0.85,
  "severity": "low|medium|high|critical",
  "location_name": "English place name or null",
  "country_iso": "2-letter ISO or null",
  "event_date": "YYYY-MM-DD or null",
  "magnitude": null,
  "depth_km": null,
  "casualties_confirmed": null,
  "casualties_estimated": null,
  "official_confirmed": false
}}"""

    config = types.GenerateContentConfig(
        response_mime_type="application/json", temperature=0.0, max_output_tokens=2048
    )

    for attempt in range(retries + 1):
        limiter.wait()
        try:
            response = client.models.generate_content(
                model=FAST_MODEL, contents=prompt, config=config
            )
            if response.text and response.text.strip():
                return {**report, **json.loads(response.text), "_model_used": FAST_MODEL}
        except Exception as e:
            if ("RESOURCE_EXHAUSTED" in str(e) or "429" in str(e)) and attempt < retries:
                wait_time = 2 ** attempt * 5
                print(f"    rate limit, retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            if "500" in str(e) or "503" in str(e):
                time.sleep(2 ** attempt)
                continue
            print(f"    Failed on [{report['source_name']}]: {e}")
            break

    return {**report, "crisis_type": "other", "confidence": 0.3,
            "severity": "low", "location_name": None, "_parse_failed": True}

# Apply binary filter
if binary_filter and crisis_reports:
    print(f"\n  Applying binary filter to {len(crisis_reports)} reports...")
    filtered = []
    for report in tqdm(crisis_reports, desc="Filtering"):
        try:
            result = binary_filter(report["cleaned_text"][:512], candidate_labels=CRISIS_LABELS)
            if result["labels"][0] == "crisis or disaster" and result["scores"][0] >= CRISIS_THRESHOLD:
                filtered.append(report)
        except:
            filtered.append(report)  # keep on failure
    print(f"  Binary filter: {len(crisis_reports)} → {len(filtered)} reports")
    crisis_reports = filtered

print(f"\n  Binary filter complete - {len(crisis_reports)} reports ready for classification")

  Loading binary filter model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

  Binary filter loaded | threshold=0.65

  Applying binary filter to 35 reports...


Filtering: 100%|██████████| 35/35 [02:20<00:00,  4.02s/it]

  Binary filter: 35 → 31 reports

  Binary filter complete - 31 reports ready for classification


In [7]:
# ======================================================================
# CELL 6: Geocoding + Batch Processing
# ======================================================================
from geopy.geocoders import Nominatim

geolocator    = Nominatim(user_agent="crisislens_v2")
_geocode_cache = {}

COUNTRY_COORDS = {
    "indonesia":   (-2.5, 118.0),    "philippines": (12.8797, 121.7740),
    "bangladesh":  (23.6850, 90.3563), "india":     (20.5937, 78.9629),
    "vietnam":     (14.0583, 108.2772), "thailand":  (15.8700, 100.9925),
    "myanmar":     (21.9162, 95.9560), "nepal":      (28.3949, 84.1240),
    "sri lanka":   (7.8731, 80.7718), "pakistan":   (30.3753, 69.3451)
}

def geocode(location_name):
    if not location_name:
        return (None, None)
    if location_name in _geocode_cache:
        return _geocode_cache[location_name]

    # Strategy 1: Full name
    try:
        result = geolocator.geocode(location_name, timeout=10)
        if result:
            coords = (result.latitude, result.longitude)
            _geocode_cache[location_name] = coords
            return coords
    except: pass

    # Strategy 2: City + Southeast Asia
    city = location_name.split(",")[0].strip()
    if city != location_name:
        try:
            result = geolocator.geocode(f"{city}, Southeast Asia", timeout=10)
            if result:
                coords = (result.latitude, result.longitude)
                _geocode_cache[location_name] = coords
                return coords
        except: pass

    # Strategy 3: Country-level fallback
    loc_lower = location_name.lower()
    for country, coords in COUNTRY_COORDS.items():
        if country in loc_lower:
            _geocode_cache[location_name] = coords
            return coords

    _geocode_cache[location_name] = (None, None)
    return (None, None)

# Load or init cache
CACHE_PATH = "/kaggle/working/classification_cache.pkl"
try:
    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    print(f"  Loaded cache with {len(cache)} entries")
except:
    cache = {}
    print("  Starting with empty cache")

# Batch classify
classified_reports = []
api_calls_this_run = 0
model_usage        = Counter()

print(f"\n  Classifying {len(crisis_reports)} reports...")
for report in tqdm(crisis_reports, desc="Classifying"):
    report_key = hash(report["raw_text"][:200])

    if report_key in cache:
        cached = cache[report_key]
        classified_reports.append(cached)
        if "_model_used" in cached:
            model_usage[cached["_model_used"]] += 1
        continue

    result = classify_and_extract(report)
    api_calls_this_run += 1

    if "_model_used" in result:
        model_usage[result["_model_used"]] += 1

    lat, lon = geocode(result.get("location_name"))
    result["latitude"]  = lat
    result["longitude"] = lon

    if not result.get("_parse_failed"):
        cache[report_key] = result
    classified_reports.append(result)

with open(CACHE_PATH, "wb") as f:
    pickle.dump(cache, f)

# Also save as list for safety
with open("/kaggle/working/classified_reports.pkl", "wb") as f:
    pickle.dump(classified_reports, f)

still_failed  = sum(1 for r in classified_reports if r.get("_parse_failed"))
geocoded_count = sum(1 for r in classified_reports if r.get("latitude") is not None)

print(f"\n  Processing Results:")
print(f"  Classified      : {len(classified_reports)}")
print(f"  API calls made  : {api_calls_this_run}")
print(f"  From cache      : {len(classified_reports) - api_calls_this_run}")
print(f"  Parse failures  : {still_failed}")
print(f"  Geocoded        : {geocoded_count}/{len(classified_reports)} ({geocoded_count/len(classified_reports)*100:.1f}%)")
print(f"  Model usage     : {dict(model_usage)}")
print(f"\n  Classification + Geocoding complete")

  Starting with empty cache

  Classifying 31 reports...


Classifying: 100%|██████████| 31/31 [11:10<00:00, 21.63s/it]


  Processing Results:
  Classified      : 31
  API calls made  : 31
  From cache      : 0
  Parse failures  : 0
  Geocoded        : 30/31 (96.8%)
  Model usage     : {'gemma-4-31b-it': 31}

  Classification + Geocoding complete


In [8]:
# ======================================================================
# CELL 7: Advanced Deduplication + Event Fusion
# ======================================================================
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from geopy.distance import geodesic
import torch

print("  Loading SBERT model...")
try:
    sbert_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
    if torch.cuda.is_available():
        sbert_model = sbert_model.to("cuda")
    print(f"  SBERT loaded on {'GPU' if torch.cuda.is_available() else 'CPU'}")
except Exception as e:
    print(f"  SBERT failed: {e}")
    sbert_model = None

CRISIS_THRESHOLDS = {
    "earthquake": {"distance_km": 500, "time_hours": 72},
    "flood":      {"distance_km": 300, "time_hours": 120},
    "storm":      {"distance_km": 800, "time_hours": 96},
    "volcano":    {"distance_km": 100, "time_hours": 240},
    "wildfire":   {"distance_km": 200, "time_hours": 48},
    "default":    {"distance_km": 250, "time_hours": 72}
}
CRISIS_SIM_THRESHOLDS = {
    "earthquake": 0.75, "flood": 0.78, "storm": 0.78,
    "volcano": 0.80,    "wildfire": 0.78, "default": 0.80
}

def get_threshold(crisis_type, key):
    return CRISIS_THRESHOLDS.get(crisis_type, CRISIS_THRESHOLDS["default"])[key]

def get_sim_threshold(crisis_type):
    return CRISIS_SIM_THRESHOLDS.get(crisis_type, CRISIS_SIM_THRESHOLDS["default"])

def are_same_event(r1, r2):
    type1, type2 = r1.get("crisis_type"), r2.get("crisis_type")
    earthquake_types = {"earthquake", "aftershock", "quake"}
    if type1 in earthquake_types and type2 in earthquake_types:
        crisis_type = "earthquake"
    elif type1 == type2:
        crisis_type = type1
    else:
        return False

    lat1, lon1 = r1.get("latitude"), r1.get("longitude")
    lat2, lon2 = r2.get("latitude"), r2.get("longitude")
    if lat1 and lon1 and lat2 and lon2:
        try:
            if geodesic((lat1, lon1), (lat2, lon2)).km > get_threshold(crisis_type, "distance_km"):
                return False
        except: pass
    else:
        c1 = r1.get("country_iso", ""); c2 = r2.get("country_iso", "")
        if c1 and c2 and c1 != c2:
            return False

    try:
        t1   = datetime.fromisoformat(r1.get("published_at", "").replace("Z", "+00:00"))
        t2   = datetime.fromisoformat(r2.get("published_at", "").replace("Z", "+00:00"))
        diff = abs((t1 - t2).total_seconds()) / 3600
        if diff > get_threshold(crisis_type, "time_hours"):
            return False
    except: pass

    return True

def are_semantically_similar(r1, r2):
    if not sbert_model:
        return False
    try:
        embs = sbert_model.encode([r1["raw_text"], r2["raw_text"]])
        sim  = cosine_similarity([embs[0]], [embs[1]])[0][0]
        return sim >= get_sim_threshold(r1.get("crisis_type", "default"))
    except:
        return False

# Stage 1: pre-filter
print(f"\n  Stage 1: Crisis-specific pre-filtering...")
stage1_pairs = [
    (i, j) for i in range(len(classified_reports))
    for j in range(i + 1, len(classified_reports))
    if are_same_event(classified_reports[i], classified_reports[j])
]
print(f"  Passed Stage 1: {len(stage1_pairs)} pairs")

# Stage 2: semantic similarity
print("  Stage 2: Semantic similarity filtering...")
final_pairs = [
    (i, j) for i, j in stage1_pairs
    if are_semantically_similar(classified_reports[i], classified_reports[j])
]
print(f"  Passed Stage 2: {len(final_pairs)} pairs")

# Union-Find event fusion
parent = list(range(len(classified_reports)))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

for i, j in final_pairs:
    union(i, j)

groups = {}
for i in range(len(classified_reports)):
    root = find(i)
    groups.setdefault(root, []).append(i)

final_events = []
for group_indices in groups.values():
    group = [classified_reports[i] for i in group_indices]
    primary = max(group, key=lambda r: r.get("trust_score", 0))
    primary["corroboration_count"] = len(group)
    primary["source_names"]        = list(set(r["source_name"] for r in group))
    primary["all_sources"]         = [r["source_name"] for r in group]
    final_events.append(primary)

final_events.sort(key=lambda e: (-e["trust_score"], -e["corroboration_count"]))

print(f"\n  Deduplication Results:")
print(f"  Input reports   : {len(classified_reports)}")
print(f"  Final events    : {len(final_events)}")
print(f"  Reduction ratio : {len(classified_reports)/len(final_events):.2f}x")
print(f"  Multi-source    : {sum(1 for e in final_events if e['corroboration_count'] > 1)}")
print(f"\n  Advanced deduplication complete - {len(final_events)} unique events")

  Loading SBERT model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  SBERT loaded on CPU

  Stage 1: Crisis-specific pre-filtering...
  Passed Stage 1: 158 pairs
  Stage 2: Semantic similarity filtering...
  Passed Stage 2: 56 pairs

  Deduplication Results:
  Input reports   : 31
  Final events    : 20
  Reduction ratio : 1.55x
  Multi-source    : 2

  Advanced deduplication complete - 20 unique events


In [9]:
# ======================================================================
# CELL 8: Confidence Scoring + Database Save
# ======================================================================
from sqlalchemy import create_engine, text as sql_text
import uuid

def calculate_system_confidence(event):
    trust_factor        = event.get("trust_score", 0.5)
    ai_confidence       = event.get("confidence", 0.5)
    n                   = event.get("corroboration_count", 1)
    corroboration_factor = min(1.0, 0.5 + (n - 1) * 0.15)
    has_official        = any(s in ["USGS","BMKG","WHO","ReliefWeb"]
                              for s in event.get("source_names", []))
    official_bonus      = 0.1 if has_official else 0.0
    geocoding_bonus     = 0.05 if event.get("latitude") is not None else 0.0

    system_confidence = (
        0.30 * trust_factor +
        0.25 * ai_confidence +
        0.25 * corroboration_factor +
        0.15 * (trust_factor + official_bonus) +
        0.05 * (ai_confidence + geocoding_bonus)
    )
    return min(1.0, system_confidence)

for event in final_events:
    event["system_confidence"] = calculate_system_confidence(event)

final_events.sort(key=lambda e: -e["system_confidence"])

print("  Setting up database...")
engine = create_engine("sqlite:////kaggle/working/crisislens_v2.db")

with engine.begin() as conn:
    conn.execute(sql_text("""
        CREATE TABLE IF NOT EXISTS reports (
            id TEXT PRIMARY KEY, source_name TEXT, source_type TEXT,
            trust_score REAL, raw_text TEXT, cleaned_text TEXT,
            language TEXT, crisis_type TEXT, severity TEXT,
            location_raw TEXT, country_iso TEXT, latitude REAL, longitude REAL,
            magnitude REAL, casualties INTEGER, official_confirmed BOOLEAN,
            classification_confidence REAL, system_confidence REAL,
            corroboration_count INTEGER, published_at TEXT, processed BOOLEAN DEFAULT 1
        )
    """))

with engine.begin() as conn:
    for event in final_events:
        conn.execute(sql_text("""
            INSERT OR REPLACE INTO reports (
                id, source_name, source_type, trust_score, raw_text, cleaned_text,
                language, crisis_type, severity, location_raw, country_iso,
                latitude, longitude, magnitude, casualties, official_confirmed,
                classification_confidence, system_confidence, corroboration_count,
                published_at, processed
            ) VALUES (
                :id, :source_name, :source_type, :trust_score, :raw_text, :cleaned_text,
                :language, :crisis_type, :severity, :location_raw, :country_iso,
                :latitude, :longitude, :magnitude, :casualties, :official_confirmed,
                :confidence, :system_confidence, :corroboration_count, :published_at, 1
            )
        """), {
            "id": str(uuid.uuid4()),
            "source_name": ", ".join(event.get("source_names", [])),
            "source_type": event.get("source_type", "unknown"),
            "trust_score": event.get("trust_score", 0.5),
            "raw_text": event.get("raw_text", ""),
            "cleaned_text": event.get("cleaned_text", ""),
            "language": event.get("language", "en"),
            "crisis_type": event.get("crisis_type", "other"),
            "severity": event.get("severity", "low"),
            "location_raw": event.get("location_name", ""),
            "country_iso": event.get("country_iso", ""),
            "latitude": event.get("latitude"),
            "longitude": event.get("longitude"),
            "magnitude": event.get("magnitude"),
            "casualties": event.get("casualties_confirmed"),
            "official_confirmed": event.get("official_confirmed", False),
            "confidence": event.get("confidence", 0.5),
            "system_confidence": event.get("system_confidence", 0.5),
            "corroboration_count": event.get("corroboration_count", 1),
            "published_at": event.get("published_at", "")
        })

with open("/kaggle/working/final_events.pkl", "wb") as f:
    pickle.dump(final_events, f)

high_conf  = [e for e in final_events if e["system_confidence"] >= 0.8]
multi_src  = [e for e in final_events if e["corroboration_count"] > 1]
geocoded   = [e for e in final_events if e.get("latitude") is not None]

print(f"\n  Final Processing Summary:")
print(f"  Total events           : {len(final_events)}")
print(f"  High confidence (≥80%) : {len(high_conf)}")
print(f"  Multi-source events    : {len(multi_src)}")
print(f"  Successfully geocoded  : {len(geocoded)} ({len(geocoded)/len(final_events)*100:.1f}%)")
print(f"  Average confidence     : {np.mean([e['system_confidence'] for e in final_events]):.3f}")

print(f"\n  Top 5 Events:")
for i, event in enumerate(final_events[:5], 1):
    srcs = ", ".join(event.get("source_names", []))
    print(f"{i}. [{event['system_confidence']:.1%}] {event.get('crisis_type','?')} "
          f"in {event.get('location_name','unknown')} ({event.get('corroboration_count',1)} src: {srcs})")

print(f"\n  Confidence scoring + database save complete")

  Setting up database...

  Final Processing Summary:
  Total events           : 20
  High confidence (≥80%) : 4
  Multi-source events    : 2
  Successfully geocoded  : 19 (95.0%)
  Average confidence     : 0.779

  Top 5 Events:
1. [89.2%] earthquake in Berau (1 src: BMKG)
2. [89.1%] other in Indonesia (11 src: GoogleNewsID)
3. [85.2%] disease in None (1 src: WHO)
4. [81.9%] other in Indonesia (2 src: GoogleNewsID)
5. [78.1%] other in Indonesia (1 src: GoogleNewsID)

  Confidence scoring + database save complete


In [16]:
# ======================================================================
# CELL 9: Role-Based Telegram Bots — Shared Core
# ======================================================================
import os, json, time, asyncio, threading, hashlib
import nest_asyncio
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

EVENTS_JSON = "/kaggle/working/final_events.json"

ROLE_TONE = {
    "citizen": (
        "You are CrisisLens AI for CITIZENS. Keep citizens safe WITHOUT causing panic. "
        "Calm, reassuring, practical tone. Give simple safety actions, not raw technical detail. "
        "Never use words like CRITICAL/URGENT unless truly critical severity."
    ),
    "tourist": (
        "You are CrisisLens AI for TOURISTS. Answer 'should I go there' style travel-safety questions. "
        "Be informative and reassuring where possible, honest about real risk where not. "
        "Suggest alternatives when relevant."
    ),
    "responder": (
        "You are CrisisLens AI for EMERGENCY RESPONDERS. Full tactical detail: severity, confidence, "
        "location precision, casualties/magnitude if available. No softening."
    ),
    "government": (
        "You are CrisisLens AI for GOVERNMENT OFFICIALS. Full detail: risk level, criticality, "
        "resource-allocation implications, confidence and corroboration. No softening."
    ),
}


def load_events():
    if os.path.exists(EVENTS_JSON):
        try:
            with open(EVENTS_JSON, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return final_events  # notebook fallback


def build_context(events, max_events=15):
    if not events:
        return "No active verified crisis events at this time."
    lines = []
    for i, e in enumerate(events[:max_events], 1):
        lines.append(
            f"{i}. {(e.get('crisis_type') or '?').title()} in {e.get('location_name','Unknown')} "
            f"({e.get('country_iso','?')}) | Severity: {(e.get('severity') or '?').upper()} | "
            f"Confidence: {(e.get('system_confidence',0) or 0):.0%} | "
            f"Sources: {', '.join(e.get('source_names') or ['Unknown'])}"
        )
    return "\n".join(lines)


def ask_gemma(question, events, role, max_retries=2):
    context = build_context(events)
    prompt = f"""{ROLE_TONE[role]}

Current verified crisis events:
{context}

User question: {question}

Answer ONLY from the verified events above. If nothing matches, say so clearly.
Be concise (3-6 sentences).

ANSWER:"""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model=DEEP_MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=500),
            )
            if resp.text and resp.text.strip():
                return resp.text.strip()
            if attempt < max_retries:
                time.sleep(3); continue
            return "⚠️ Empty response from model — try again in a moment."
        except Exception as e:
            if attempt < max_retries:
                time.sleep(3); continue
            return f"⚠️ AI request failed: {e}"


def event_id(e):
    key = f"{e.get('crisis_type')}|{e.get('location_name')}|{e.get('event_date')}"
    return hashlib.md5(key.encode()).hexdigest()


def format_alert(e, role):
    conf = e.get("system_confidence", 0) or 0
    crisis = (e.get("crisis_type") or "?").title()
    loc = e.get("location_name", "Unknown")
    if role == "citizen":
        return (f"📍 Update: A {crisis} has been reported near {loc}. "
                f"Please stay alert and follow local guidance.")
    detail = []
    if e.get("magnitude"): detail.append(f"Magnitude: {e['magnitude']}")
    if e.get("casualties_estimated"): detail.append(f"Est. Affected: {e['casualties_estimated']}")
    detail_str = " | " + " | ".join(detail) if detail else ""
    return (f"🚨 {crisis} — {loc}\n"
            f"Severity: {(e.get('severity') or '?').upper()} | Confidence: {conf:.0%}{detail_str}\n"
            f"Sources: {', '.join(e.get('source_names') or ['Unknown'])}")


class RoleTelegramBot:
    def __init__(self, role, token, chat_id=None, push_enabled=False, push_interval=300):
        self.role = role
        self.token = token
        self.chat_id = chat_id
        self.push_enabled = push_enabled and bool(chat_id)
        self.push_interval = push_interval
        self.seen_ids = set()

    async def start_cmd(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        await update.message.reply_text(f"👋 CrisisLens AI — {self.role.title()} bot ready. Ask me anything.")

    async def handle_message(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        question = update.message.text
        await context.bot.send_chat_action(chat_id=update.effective_chat.id, action="typing")
        answer = ask_gemma(question, load_events(), self.role)
        await update.message.reply_text(answer)

    async def alert_loop(self, application):
        # seed with current events so we don't spam alerts for old data on first launch
        self.seen_ids = {event_id(e) for e in load_events()}
        while True:
            await asyncio.sleep(self.push_interval)
            try:
                events = load_events()
                new_events = [e for e in events if event_id(e) not in self.seen_ids]
                for e in new_events:
                    self.seen_ids.add(event_id(e))
                    await application.bot.send_message(chat_id=self.chat_id, text=format_alert(e, self.role))
            except Exception as ex:
                print(f"[{self.role}] alert loop error: {ex}")

    def run_background(self):
        def _run():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            nest_asyncio.apply(loop)

            app = Application.builder().token(self.token).build()
            app.add_handler(CommandHandler("start", self.start_cmd))
            app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, self.handle_message))

            if self.push_enabled:
                self.seen_ids = {event_id(e) for e in load_events()}
                loop.create_task(self.alert_loop(app))  # scheduled on the loop directly, not via post_init

            print(f"🚀 {self.role.title()} bot starting (push={'on' if self.push_enabled else 'off'})...")
            app.run_polling(allowed_updates=Update.ALL_TYPES, stop_signals=None)

        t = threading.Thread(target=_run, daemon=True)
        t.start()
        return t


print("✅ Shared bot core loaded. Launch each role bot below, one at a time.")

✅ Shared bot core loaded. Launch each role bot below, one at a time.


No error handlers are registered, logging exception.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 161, in network_retry_loop
    await do_action()
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_utils/networkloop.py", line 154, in do_action
    action_cb_task.result()
  File "/usr/lib/python3.12/asyncio/futures.py", line 202, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 340, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 680, in get_updates
    updates = await super().get_updates(
              ^^^^^^^^^^^^^^^^

In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

def load_secret(name, required=False):
    try:
        return user_secrets.get_secret(name)
    except Exception:
        print(f"⚠️ '{name}' not found — check Add-ons → Secrets and make sure it's attached.")
        return None

for key in ["CITIZEN_TELEGRAM_BOT_TOKEN", "TOURIST_TELEGRAM_BOT_TOKEN",
            "RESPONDER_TELEGRAM_BOT_TOKEN", "GOVERNMENT_TELEGRAM_BOT_TOKEN"]:
    val = load_secret(key)
    if val:
        os.environ[key] = val
        print(f"✅ {key} loaded")

✅ CITIZEN_TELEGRAM_BOT_TOKEN loaded
✅ TOURIST_TELEGRAM_BOT_TOKEN loaded
✅ RESPONDER_TELEGRAM_BOT_TOKEN loaded
✅ GOVERNMENT_TELEGRAM_BOT_TOKEN loaded


In [15]:
citizen_bot = RoleTelegramBot(
    role="citizen",
    token="PASTE_THE_TOKEN_BOTFATHER_GAVE_YOU_HERE",
    chat_id=os.environ.get("TELEGRAM_CHAT_ID"),
    push_enabled=True,
    push_interval=300,
)
citizen_bot.run_background()

<Thread(Thread-8 (_run), started daemon 140001597650496)>

🚀 Citizen bot starting (push=on)...


Network Retry Loop (Bootstrap Initialize Application): Invalid token. Aborting retry loop.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/_bot.py", line 865, in initialize
    await self.get_me()
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 2040, in get_me
    return await super().get_me(
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/_bot.py", line 1005, in get_me
    result = await self._post(
             ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/_bot.py", line 712, in _post
    return await self._do_post(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 378, in _do_post
    return await super()._do_post(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/_bot.py", line 741, in _do_post
    result = await request.post(
          

In [9]:
# ======================================================================
# CELL 9: Gemma AI Crisis Assistant (Simple Inline GUI — for demo/screen recording)
# ======================================================================
import time
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def build_events_context(events, max_events=15):
    if not events:
        return "No active verified crisis events at this time."
    lines = []
    for i, e in enumerate(events[:max_events], 1):
        crisis_type = (e.get("crisis_type") or "Unknown").title()
        location = e.get("location_name", "Unknown Location")
        country = e.get("country_iso", "?")
        severity = (e.get("severity") or "unknown").upper()
        confidence = e.get("system_confidence", 0) or 0
        sources = ", ".join(e.get("source_names") or ["Unknown"])
        extra = ""
        if e.get("crisis_type") == "earthquake" and e.get("magnitude"):
            extra = f" | Magnitude: {e.get('magnitude')}"
        elif e.get("casualties_estimated"):
            extra = f" | Est. Affected: {e.get('casualties_estimated')}"
        lines.append(
            f"{i}. {crisis_type} in {location} ({country}) | Severity: {severity} | "
            f"Confidence: {confidence:.0%}{extra} | Sources: {sources}"
        )
    return "\n".join(lines)


def ask_gemma(question, events, max_retries=2):
    context = build_events_context(events)
    prompt = f"""You are CrisisLens AI, a disaster intelligence assistant for South and Southeast Asia.

Current verified crisis events:
{context}

User question: {question}

Answer ONLY based on the verified events above. If no data covers the question,
say so clearly instead of guessing. Be concise (3-6 sentences), factual, and actionable.

ANSWER:"""

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model=DEEP_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=800),
            )

            if resp.text and resp.text.strip():
                return resp.text.strip()

            # Empty response — dig into WHY instead of guessing
            reason = "unknown"
            try:
                if resp.candidates:
                    cand = resp.candidates[0]
                    reason = getattr(cand, "finish_reason", "unknown")
                    if getattr(cand, "safety_ratings", None):
                        blocked = [r for r in cand.safety_ratings if getattr(r, "blocked", False)]
                        if blocked:
                            reason = f"SAFETY_BLOCK: {[r.category for r in blocked]}"
                if getattr(resp, "prompt_feedback", None):
                    reason = f"{reason} | prompt_feedback={resp.prompt_feedback}"
            except Exception:
                pass

            print(f"    [debug] Empty response on attempt {attempt+1}, reason: {reason}")

            if attempt < max_retries:
                time.sleep(3)
                continue
            return f"⚠️ Empty response from model after {max_retries+1} attempts. Reason: {reason}"

        except Exception as e:
            print(f"    [debug] Exception on attempt {attempt+1}: {e}")
            if attempt < max_retries:
                time.sleep(3)
                continue
            return f"⚠️ AI request failed after {max_retries+1} attempts: {e}"


# --- Simple chat GUI ---
question_box = widgets.Text(
    placeholder="Ask about the current crisis situation...",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask CrisisLens AI 🚨", button_style="danger")
output_area = widgets.Output()
conversation = []


def render_conversation(thinking=False):
    with output_area:
        clear_output(wait=True)
        for speaker, text in conversation:
            if speaker == "You":
                display(HTML(f"<div style='text-align:right;margin:6px;'><b>🧑 You:</b> {text}</div>"))
            else:
                display(HTML(
                    f"<div style='text-align:left;margin:6px;background:#1e1e1e;color:#eee;"
                    f"padding:10px;border-radius:8px;'><b>🤖 CrisisLens AI:</b><br>{text}</div>"
                ))
        if thinking:
            display(HTML("<i>🧠 Thinking...</i>"))


def on_ask_clicked(b):
    question = question_box.value.strip()
    if not question:
        return
    question_box.value = ""
    conversation.append(("You", question))
    render_conversation(thinking=True)

    answer = ask_gemma(question, final_events)
    conversation.append(("AI", answer.replace(chr(10), "<br>")))
    render_conversation(thinking=False)


ask_button.on_click(on_ask_clicked)
try:
    question_box.on_submit(on_ask_clicked)  # Enter key shortcut, if supported by this ipywidgets version
except Exception:
    pass

print(f"🤖 CrisisLens AI Assistant ready — grounded in {len(final_events)} verified events")
display(widgets.HBox([question_box, ask_button]))
display(output_area)

🤖 CrisisLens AI Assistant ready — grounded in 25 verified events


/tmp/ipykernel_58/4165056417.py:127: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  question_box.on_submit(on_ask_clicked)  # Enter key shortcut, if supported by this ipywidgets version


Output()

In [10]:
# ======================================================================
# CELL 10: Live Crisis Map (Inline, lightweight, detailed)
# ======================================================================
import os
import json
import time
import folium
from folium.plugins import MarkerCluster
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

MAP_EVENTS_JSON = "/kaggle/working/final_events.json"  # falls back to in-memory final_events

CRISIS_COLORS = {
    "flood": "blue",
    "earthquake": "red",
    "volcano": "orange",
    "wildfire": "darkred",
    "landslide": "beige",
    "storm": "purple",
    "disease": "green",
}

SEVERITY_RADIUS = {"low": 6, "medium": 9, "high": 12, "critical": 15}


def get_current_events():
    """Prefer the exported JSON if present (e.g. from the integration script),
    otherwise fall back to whatever's in memory as final_events."""
    if os.path.exists(MAP_EVENTS_JSON):
        try:
            with open(MAP_EVENTS_JSON, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return final_events


def build_crisis_map(events):
    geo_events = [e for e in events if e.get("latitude") and e.get("longitude")]

    center_lat = geo_events[0]["latitude"] if geo_events else -2.5
    center_lon = geo_events[0]["longitude"] if geo_events else 118.0

    # Lighter tile layer — fewer visual assets to load than the dark theme
    m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="CartoDB positron")

    # Clustering = fewer DOM nodes rendered at once = lighter in-browser,
    # and gives you a free "N events here" detail at a glance when zoomed out.
    cluster = MarkerCluster(name="Crisis Events").add_to(m)

    for e in geo_events:
        crisis = (e.get("crisis_type") or "unknown").lower()
        color = CRISIS_COLORS.get(crisis, "gray")
        conf = e.get("system_confidence", 0) or 0
        severity = (e.get("severity") or "unknown").lower()
        radius = SEVERITY_RADIUS.get(severity, 8)
        sources = ", ".join(e.get("source_names") or ["Unknown"])
        verified = "✅ Official" if e.get("official_confirmed") else "⚪ Unverified"
        trust = e.get("trust_score")

        extra_rows = ""
        if crisis == "earthquake" and e.get("magnitude"):
            extra_rows += f"<b>Magnitude:</b> {e.get('magnitude')}<br>"
        if e.get("casualties_estimated"):
            extra_rows += f"<b>Est. Affected:</b> {e.get('casualties_estimated')}<br>"
        if e.get("event_date"):
            extra_rows += f"<b>Date:</b> {e.get('event_date')}<br>"
        if trust is not None:
            extra_rows += f"<b>Trust Score:</b> {trust:.0%}<br>"

        popup_html = f"""
        <div style="font-family:sans-serif;font-size:13px;">
        <b style="font-size:14px;">{crisis.title()}</b><br>
        📍 {e.get('location_name','Unknown')}, {e.get('country_iso','?')}<br>
        🎯 <b>Confidence:</b> {conf:.0%}<br>
        ⚠️ <b>Severity:</b> {severity.upper()}<br>
        {extra_rows}
        📰 <b>Sources:</b> {sources}<br>
        {verified}
        </div>
        """

        folium.CircleMarker(
            location=[e["latitude"], e["longitude"]],
            radius=radius,
            color=color,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=270),
            tooltip=f"{crisis.title()} — {e.get('location_name','')} ({conf:.0%})",
        ).add_to(cluster)

    legend_items = "".join(
        f'<div><span style="background:{c};width:10px;height:10px;display:inline-block;'
        f'border-radius:50%;margin-right:6px;"></span>{t.title()}</div>'
        for t, c in CRISIS_COLORS.items()
    )
    legend_html = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
                background: white; color: #222; padding: 10px 14px; border-radius: 8px;
                font-size: 12px; border: 1px solid #ccc; box-shadow: 0 1px 4px rgba(0,0,0,0.2);">
        <b>Crisis Type</b>{legend_items}
        <div style="margin-top:6px;font-size:11px;color:#666;">Marker size = severity</div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    return m, geo_events, len(events)


def render_stats(events, geo_events):
    total = len(events)
    high_conf = sum(1 for e in events if (e.get("system_confidence", 0) or 0) >= 0.8)
    verified = sum(1 for e in events if e.get("official_confirmed"))
    avg_conf = (sum((e.get("system_confidence", 0) or 0) for e in events) / total) if total else 0

    stats_html = f"""
    <div style="display:flex;gap:16px;margin-bottom:8px;font-family:sans-serif;">
        <div style="background:#1e1e1e;color:#eee;padding:8px 16px;border-radius:8px;">
            <b>{total}</b><br><span style="font-size:11px;color:#aaa;">Total Events</span>
        </div>
        <div style="background:#1e1e1e;color:#eee;padding:8px 16px;border-radius:8px;">
            <b>{len(geo_events)}</b><br><span style="font-size:11px;color:#aaa;">Mapped</span>
        </div>
        <div style="background:#1e1e1e;color:#eee;padding:8px 16px;border-radius:8px;">
            <b>{high_conf}</b><br><span style="font-size:11px;color:#aaa;">High Confidence</span>
        </div>
        <div style="background:#1e1e1e;color:#eee;padding:8px 16px;border-radius:8px;">
            <b>{verified}</b><br><span style="font-size:11px;color:#aaa;">Verified</span>
        </div>
        <div style="background:#1e1e1e;color:#eee;padding:8px 16px;border-radius:8px;">
            <b>{avg_conf:.0%}</b><br><span style="font-size:11px;color:#aaa;">Avg Confidence</span>
        </div>
    </div>
    """
    return HTML(stats_html)


def render_top_events(events):
    top = sorted(events, key=lambda e: (e.get("system_confidence", 0) or 0), reverse=True)[:5]
    rows = "".join(
        f"<li>[{(e.get('system_confidence',0) or 0):.0%}] "
        f"<b>{(e.get('crisis_type') or '?').title()}</b> in "
        f"{e.get('location_name','Unknown')} — {(e.get('severity') or '?').upper()}</li>"
        for e in top
    )
    return HTML(f"<div style='font-family:sans-serif;font-size:13px;'><b>Top 5 by confidence</b><ul>{rows}</ul></div>")


# --- Widgets ---
map_output = widgets.Output()
refresh_button = widgets.Button(description="🔄 Refresh Map", button_style="info")
status_label = widgets.Label(value="")


def render_all():
    events = get_current_events()
    m, geo_events, total = build_crisis_map(events)
    with map_output:
        clear_output(wait=True)
        display(render_stats(events, geo_events))
        display(m)
        display(render_top_events(events))
    status_label.value = f"Last updated {time.strftime('%H:%M:%S')}"


refresh_button.on_click(lambda b: render_all())

render_all()  # initial render
display(widgets.HBox([refresh_button, status_label]))
display(map_output)

Output()

In [11]:
# ======================================================================
# CELL 10: Telegram Alert System
# ======================================================================
import nest_asyncio
nest_asyncio.apply()

async def send_telegram_alert(event):
    from telegram import Bot
    bot_token = os.environ.get("TELEGRAM_BOT_TOKEN")
    chat_id   = os.environ.get("TELEGRAM_CHAT_ID")
    if not bot_token or not chat_id:
        return False
    try:
        bot  = Bot(token=bot_token)
        conf = event.get("system_confidence", 0)
        srcs = ", ".join(event.get("source_names", []))
        emoji_map = {"low":"🟡","medium":"🟠","high":"🔴","critical":"🚨"}
        emoji = emoji_map.get(event.get("severity","low"), " ")

        briefing_prompt = f"""You are a crisis analyst. Write a 3-part briefing for emergency responders:

Event: {event.get('crisis_type')} in {event.get('location_name')}
Severity: {event.get('severity')} | Confidence: {conf:.0%}
Sources: {srcs}
Text: {event.get('raw_text','')[:300]}

Format exactly:
SITUATION: [1 sentence facts]
IMPACT: [1 sentence current impact]
ACTION: [1 sentence recommended action]"""

        try:
            resp = client.models.generate_content(
                model=DEEP_MODEL, contents=briefing_prompt,
                config=types.GenerateContentConfig(temperature=0.1, max_output_tokens=400)
            )
            briefing = resp.text.strip() if resp.text else "Briefing unavailable"
        except:
            briefing = "Briefing generation failed"

        message = (f"{emoji} *CrisisLens Alert*\n\n"
                   f"*{event.get('crisis_type','?').upper()}* — {event.get('location_name','Unknown')}\n"
                   f"Confidence: {conf:.0%} | Sources: {srcs}\n"
                   f"Severity: {event.get('severity','unknown')}\n\n"
                   f"{briefing}\n\n"
                   f"_Automated by CrisisLens AI 2.0_")

        await bot.send_message(chat_id=chat_id, text=message, parse_mode="Markdown")
        return True
    except Exception as e:
        print(f"  Telegram alert failed: {e}")
        return False

# Send alerts
print("📱 Sending Telegram alerts...")
ALERT_THRESHOLD = 0.65
events_to_alert = [e for e in final_events if e.get("system_confidence", 0) >= ALERT_THRESHOLD]

if not os.environ.get("TELEGRAM_BOT_TOKEN"):
    print("  TELEGRAM_BOT_TOKEN not set - alerts disabled")
elif not events_to_alert:
    print(f"  No events above threshold ({ALERT_THRESHOLD})")
else:
    import asyncio
    sent = 0
    for event in events_to_alert[:5]:  # max 5 alerts
        try:
            success = asyncio.get_event_loop().run_until_complete(send_telegram_alert(event))
            if success:
                print(f"    Sent: {event.get('crisis_type')} in {event.get('location_name')}")
                sent += 1
        except Exception as e:
            print(f"    Failed: {e}")
    print(f"\n  Telegram alerts complete — {sent}/{len(events_to_alert)} sent")

📱 Sending Telegram alerts...
    Sent: other in Indonesia
    Sent: earthquake in Tahuna, Sangihe Islands
    Sent: disease in None
    Sent: other in Indonesia
    Sent: earthquake in Bali

  Telegram alerts complete — 5/25 sent
